# Machine Learning Assignment - 2 (Course: S1-26_AIMLZG565)

### Name: Rishabh Singh
### Id: 2026ak05163

# 1. Introduction and Problem Statement

The objective of this assignment is to develop and evaluate multiple machine learning classification models using the Online Shoppers Purchasing Intention dataset.

The task is to predict whether an online shopping session will result in a purchase based on visitor behaviour and session-related attributes.

We implement five classification algorithms:

1. Logistic Regression
2. Decision Tree Classifier
3. K-Nearest Neighbors (KNN)
4. Gaussian Naive Bayes
5. Support Vector Machine (SVM)

The models are evaluated using Accuracy, AUC, Precision, Recall, F1 Score, and Matthews Correlation Coefficient (MCC).

In [3]:
!pip install ucimlrepo
!pip install scikit-learn

# 2. Dataset Description

The Online Shoppers Purchasing Intention dataset is obtained from the UCI Machine Learning Repository.

The dataset contains 12,330 instances and 18 columns, consisting of 17 input features and one target variable (`Revenue`).

The target variable indicates whether a shopping session resulted in a purchase:

- `True`: Purchase occurred.
- `False`: No purchase occurred.

The dataset contains numerical, categorical, and Boolean features. No missing values were found during initial inspection.

The target distribution is imbalanced, with approximately 84.5% non-purchase sessions and 15.5% purchase sessions.

URL: https://archive.ics.uci.edu/dataset/468/online+shoppers+purchasing+intention+dataset

In [5]:
import pandas as pd
from ucimlrepo import fetch_ucirepo

# Fetch the official UCI dataset
dataset = fetch_ucirepo(id=468)

# Combine features and target
X = dataset.data.features
y = dataset.data.targets

df = pd.concat([X, y], axis=1)

# 1. Dataset dimensions
print("Dataset Shape:", df.shape)

# 2. First five records
print("\nFirst Five Records:")
print(df.head())

# 3. Column names and data types
print("\nDataset Information:")
df.info()

# 4. Missing values
print("\nMissing Values:")
print(df.isnull().sum())

# 5. Target distribution
print("\nTarget Distribution:")
print(df["Revenue"].value_counts())

# 6. Target distribution in percentage
print("\nTarget Distribution (%):")
print(df["Revenue"].value_counts(normalize=True) * 100)

# 7. Unique values in each feature
print("\nUnique Values:")
print(df.nunique())

# 8. Save original dataset
df.to_csv("online_shoppers_intention.csv", index=False)

Dataset Shape: (12330, 18)

First Five Records:
   Administrative  Administrative_Duration  Informational  \
0               0                      0.0              0   
1               0                      0.0              0   
2               0                      0.0              0   
3               0                      0.0              0   
4               0                      0.0              0   

   Informational_Duration  ProductRelated  ProductRelated_Duration  \
0                     0.0               1                 0.000000   
1                     0.0               2                64.000000   
2                     0.0               1                 0.000000   
3                     0.0               2                 2.666667   
4                     0.0              10               627.500000   

   BounceRates  ExitRates  PageValues  SpecialDay Month  OperatingSystems  \
0         0.20       0.20         0.0         0.0   Feb                 1   
1         

# 3. Data Preprocessing and Train-Test Split

The input features and target variable were separated before model development.

Categorical features were identified separately from numerical features. Integer-coded identifiers such as Browser, Region, OperatingSystems, and TrafficType were treated as categorical variables rather than continuous numerical measurements.

The dataset was divided into training and testing subsets using an 80:20 stratified split.

A random state of 42 was used to ensure reproducibility.

Stratified sampling was selected to maintain approximately the same target-class proportions in both subsets, which is important because the dataset is imbalanced.

The resulting datasets contain:

- Training instances: 9,864
- Testing instances: 2,466

The test dataset was exported as `test_data.csv` for subsequent evaluation through the Streamlit application.

Preprocessing transformations will be fitted exclusively on the training data to prevent data leakage.

In [7]:
from sklearn.model_selection import train_test_split

# Separate input features and target
X = df.drop(columns=["Revenue"])
y = df["Revenue"]

# Perform stratified train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Display dataset sizes
print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True) * 100)

Training features: (9864, 17)
Testing features: (2466, 17)

Training target distribution:
Revenue
False    84.529603
True     15.470397
Name: proportion, dtype: float64

Testing target distribution:
Revenue
False    84.509327
True     15.490673
Name: proportion, dtype: float64


In [8]:
# Combine test features and actual target
test_data = X_test.copy()
test_data["Revenue"] = y_test

# Save test dataset
test_data.to_csv("test_data.csv", index=False)

print("Test dataset saved successfully.")
print("Shape:", test_data.shape)

Test dataset saved successfully.
Shape: (2466, 18)


## 3. Data Preprocessing

The dataset contains numerical, categorical, and binary features.
Numerical features are standardized, categorical features are one-hot
encoded, and the binary feature is passed through unchanged.

All preprocessing transformations will be fitted on the training data
only to prevent data leakage.

In [10]:
numeric_features = [
    "Administrative",
    "Administrative_Duration",
    "Informational",
    "Informational_Duration",
    "ProductRelated",
    "ProductRelated_Duration",
    "BounceRates",
    "ExitRates",
    "PageValues",
    "SpecialDay"
]

categorical_features = [
    "Month",
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType",
    "VisitorType"
]

binary_features = ["Weekend"]

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

def create_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric_features),
            (
                "cat",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                ),
                categorical_features
            ),
            ("bin", "passthrough", binary_features)
        ],
        remainder="drop"
    )

In [12]:
preprocessor = create_preprocessor()

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Original training shape:", X_train.shape)
print("Processed training shape:", X_train_processed.shape)

print("Original testing shape:", X_test.shape)
print("Processed testing shape:", X_test_processed.shape)

Original training shape: (9864, 17)
Processed training shape: (9864, 74)
Original testing shape: (2466, 17)
Processed testing shape: (2466, 74)


## 4. Classification Models

Five classification algorithms are implemented:

1. Logistic Regression
2. Decision Tree Classifier
3. K-Nearest Neighbors
4. Gaussian Naive Bayes
5. Support Vector Machine

Each model uses the same preprocessing strategy and training dataset.

In [14]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),

    "Decision Tree": DecisionTreeClassifier(random_state=42),

    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5),

    "Gaussian Naive Bayes": GaussianNB(),

    "Support Vector Machine": SVC(
        kernel="rbf",
        probability=True,
        random_state=42
    )
}

### Training the Classification Models

A separate pipeline is created for each classifier. Each pipeline independently
fits its preprocessing transformations on the training data before training
the classifier.

In [16]:
trained_models = {}

for model_name, classifier in models.items():

    pipeline = Pipeline(steps=[
        ("preprocessor", create_preprocessor()),
        ("classifier", classifier)
    ])

    pipeline.fit(X_train, y_train)

    trained_models[model_name] = pipeline

    print(f"{model_name} trained successfully.")

Logistic Regression trained successfully.
Decision Tree trained successfully.
K-Nearest Neighbors trained successfully.
Gaussian Naive Bayes trained successfully.


/home/cloud/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Support Vector Machine trained successfully.


## 5. Model Evaluation

The trained classifiers are evaluated using the held-out test dataset.

The following metrics are used:

1. Accuracy
2. Area Under the ROC Curve (AUC)
3. Precision
4. Recall
5. F1-score
6. Matthews Correlation Coefficient (MCC)

In [18]:
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)

results = []

for model_name, model in trained_models.items():

    # Generate class predictions
    y_pred = model.predict(X_test)

    # Generate probability scores for the positive class
    y_prob = model.predict_proba(X_test)[:, 1]

    metrics = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_prob),
        "Precision": precision_score(y_test, y_pred, pos_label=True, zero_division=0),
        "Recall": recall_score(y_test, y_pred, pos_label=True, zero_division=0),
        "F1": f1_score(y_test, y_pred, pos_label=True, zero_division=0),
        "MCC": matthews_corrcoef(y_test, y_pred)
    }

    results.append(metrics)

results_df = pd.DataFrame(results)

results_df = results_df.set_index("Model")

results_df = results_df.round(4)

print(results_df)

                        Accuracy     AUC  Precision  Recall      F1     MCC
Model                                                                      
Logistic Regression       0.8812  0.8830     0.7405  0.3586  0.4832  0.4610
Decision Tree             0.8646  0.7435     0.5622  0.5681  0.5651  0.4849
K-Nearest Neighbors       0.8715  0.7572     0.6419  0.3848  0.4812  0.4307
Gaussian Naive Bayes      0.2733  0.7330     0.1727  0.9738  0.2934  0.1292
Support Vector Machine    0.8873  0.8817     0.7063  0.4660  0.5615  0.5142


In [19]:
from sklearn.metrics import confusion_matrix

for model_name, model in trained_models.items():

    y_pred = model.predict(X_test)

    cm = confusion_matrix(y_test, y_pred)

    print(f"\n{model_name}")
    print("Confusion Matrix:")
    print(cm)


Logistic Regression
Confusion Matrix:
[[2036   48]
 [ 245  137]]

Decision Tree
Confusion Matrix:
[[1915  169]
 [ 165  217]]

K-Nearest Neighbors
Confusion Matrix:
[[2002   82]
 [ 235  147]]

Gaussian Naive Bayes
Confusion Matrix:
[[ 302 1782]
 [  10  372]]

Support Vector Machine
Confusion Matrix:
[[2010   74]
 [ 204  178]]


## 6. Saving Trained Models

The five trained classification pipelines are saved using Joblib.
Each saved pipeline contains both the fitted preprocessing transformations
and the trained classifier.

These artifacts will be loaded by the Streamlit application for evaluation
on uploaded test data.

In [21]:
import os
import joblib

# Create a directory to store trained models
os.makedirs("model", exist_ok=True)

# Save each trained pipeline
for model_name, model in trained_models.items():

    filename = model_name.lower().replace(" ", "_") + ".joblib"
    filepath = os.path.join("model", filename)

    joblib.dump(model, filepath)

    print(f"Saved: {filepath}")

Saved: model/logistic_regression.joblib
Saved: model/decision_tree.joblib
Saved: model/k-nearest_neighbors.joblib
Saved: model/gaussian_naive_bayes.joblib
Saved: model/support_vector_machine.joblib


In [22]:
results_df.to_csv("model_evaluation_results.csv")

In [23]:
import sklearn
import pandas
import numpy
import joblib

print("scikit-learn:", sklearn.__version__)
print("pandas:", pandas.__version__)
print("numpy:", numpy.__version__)
print("joblib:", joblib.__version__)

scikit-learn: 1.9.1
pandas: 2.2.3
numpy: 2.5.4
joblib: 1.4.2
